# 2-1. 시계열 트렌드 분석 (Time Series EDA)

경기도 카드소비 x 날씨 데이터 EDA 2단계.
목표: 일별/월별 매출 추이, 요일 패턴, 계절 패턴을 확인하고 데이터 누락 구간이 있는지 점검한다.
분류등급(필수/애매/불필요)별 비교를 유지한다.

참고: 요일은 원본 `day` 코드(1~7, 정확한 매핑 불명) 대신 `date`(datetime) 컬럼에서
직접 계산한다 — 해석의 모호함이 없는 안전한 방법이다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

system = platform.system()
if system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
%matplotlib inline

## 2-1-0. 데이터 로드

In [ ]:
from pathlib import Path

# src/analysis/ 에서 실행하는 걸 기준으로 프로젝트 루트까지 2단계 상위 경로.
CATEGORY_DIR = Path("../../data/processed/consume_weather_by_category")

# 순회할 카테고리 분류 등급 리스트를 정의
GROUP_ORDER = ["필수", "애매", "불필요"]

# 각 그룹의 데이터프레임을 임시 저장할 딕셔너리를 생성
group_dfs = {}

# "필수" -> "애매" -> "불필요" 파일을 순서대로 읽어와 '분류등급' 컬럼을 추가한 뒤 딕셔너리에 보관
for group in GROUP_ORDER:
    g_df = pd.read_parquet(CATEGORY_DIR / f"{group}.parquet")
    g_df["분류등급"] = group
    group_dfs[group] = g_df

# 3개의 데이터프레임을 하나로 수직 통합하여 통합 데이터프레임(df)을 생성
df = pd.concat(group_dfs.values(), ignore_index=True)

# [데이터 타입 정형화]
df["date"] = pd.to_datetime(df["date"])

df["요일"] = df["date"].dt.day_name()
# 딕셔너리(.map())를 활용하여 직관적인 한글 요일명('월', '화'...)으로 변환
df["요일_한글"] = df["date"].dt.dayofweek.map({0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"})
# [연월 주기 추출]
df["연월"] = df["date"].dt.to_period("M")

print("shape:", df.shape)
print("기간:", df["date"].min(), "~", df["date"].max())
df.head()

## 2-1-1. 일별 총매출 추이 (분류등급별)

카드 소비 일별 데이터는 요일 효과 때문에 노이즈가 크므로, 원본 라인과
7일 이동평균(rolling mean)을 같이 그려서 추세를 더 명확히 본다.

In [ ]:
daily = df.groupby(["date", "분류등급"])["매출금액"].sum().reset_index()

# 일별/분류등급별 매출금액 집계

fig, ax = plt.subplots(figsize=(14, 5))
for g in GROUP_ORDER:
    sub = daily[daily["분류등급"] == g].sort_values("date")
    ax.plot(sub["date"], sub["매출금액"], alpha=0.25, linewidth=0.8)
    # 요일 효과나 일별 노이즈를 제거하고 전반적인 트렌드를 보기 위해 7일 이동평균을 계산
    ax.plot(sub["date"], sub["매출금액"].rolling(7, min_periods=1).mean(), label=f"{g} (7일 이동평균)", linewidth=1.8)

ax.set_title("일별 총매출 추이 (분류등급별, 옅은선=원본/굵은선=7일 이동평균)")
ax.set_xlabel("날짜")
ax.set_ylabel("매출금액 합계")
ax.legend()
# 시계열이 길 때 x축 눈금이 빽빽하게 겹치는 것을 막기 위해 3개월 간격(interval=3)으로 주요 눈금을 설정
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
# X축 눈금에 표시될 날짜 형식을 '연도-월(YYYY-MM)' 형태로 지정
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2-1-2. 도시별 일별 추이 (필수 그룹 기준)

9개 도시를 한 그래프에 다 겹치면 알아보기 어려우므로, 분석 핵심인 '필수' 그룹만 도시별로 비교한다.

In [ ]:
essential = df[df["분류등급"] == "필수"]

# 추출한 필수 등급 데이터 내에서 'date'와 '도시'를 기준으로 그룹을 묶어 매출금액의 총합계
daily_city = essential.groupby(["date", "도시"])["매출금액"].sum().reset_index()

# 시각화 도화지 준비 및 도시별 시계열 그래프 그리기
fig, ax = plt.subplots(figsize=(14, 6))
for city in sorted(daily_city["도시"].unique()):
    # 현재 순회 중인 도시에 해당하는 데이터만 필터링한 후, 시간 흐름대로 정렬
    sub = daily_city[daily_city["도시"] == city].sort_values("date")
    # 도시별 요일 효과와 급격한 변동을 정돈하여 추세를 보기 위해 7일 이동평균선만
    ax.plot(sub["date"], sub["매출금액"].rolling(7, min_periods=1).mean(), label=city, linewidth=1.2)

ax.set_title("도시별 일별 매출금액 추이 (필수 그룹, 7일 이동평균)")
ax.set_xlabel("날짜")
ax.set_ylabel("매출금액 합계")
ax.legend(loc="upper left", ncol=3, fontsize=8)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2-1-3. 요일 패턴

평일/주말 소비 패턴 차이를 분류등급별로 비교한다.
매출금액은 왜도가 크므로 로그변환된 값으로 비교한다.

In [ ]:
weekday_order = ["월", "화", "수", "목", "금", "토", "일"]

# 요일과 분류등급을 기준으로 매출금액의 특징을 추출하고 데이터 구조를 변형
weekday_stats = (
    df.groupby(["요일_한글", "분류등급"])["매출금액"]
    # 각 그룹별 데이터에 로그 변환(np.log1p)을 적용한 후 평균값 구하기
    .apply(lambda x: np.log1p(x).mean())
    # 인덱스를 초기화하여 데이터프레임 형태로 변환
    .reset_index()
    .pivot(index="요일_한글", columns="분류등급", values="매출금액")
    .reindex(weekday_order)
)

# 요일 순서 정의 및 요일별 통계량 집계
# 시각화 도화지 준비 및 다중 막대그래프 생성
fig, ax = plt.subplots(figsize=(8, 5))
weekday_stats[GROUP_ORDER].plot(kind="bar", ax=ax)
ax.set_title("요일별 평균 log(매출금액+1) (분류등급별)")
ax.set_xlabel("요일")
ax.set_ylabel("평균 log(매출금액 + 1)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2-1-4. 월별(계절) 패턴

연도 효과를 제거하고 순수 계절성만 보기 위해, 연도-월이 아니라 '월'(1~12)만 기준으로
평균을 낸다 (2022~2025년 데이터를 월별로 합쳐서 봄).

In [ ]:
# '월' 파생 변수 생성 및 월별 통계량 집계
# datetime 형식인 'date' 컬럼에서 '월(1~12)' 정보만 정수형으로 추출하여 새로운 컬럼 만들기
df["월"] = df["date"].dt.month

monthly_stats = (
    df.groupby(["월", "분류등급"])["매출금액"]
    .apply(lambda x: np.log1p(x).mean())
    .reset_index()
    .pivot(index="월", columns="분류등급", values="매출금액")
)



fig, ax = plt.subplots(figsize=(10, 5))
# 사전에 정의해둔 분류 등급 순서("필수" -> "애매" -> "불필요")에 따라 순회하며 꺾은선그래프 그리기
for g in GROUP_ORDER:
    ax.plot(monthly_stats.index, monthly_stats[g], marker="o", label=g)

ax.set_title("월별 평균 log(매출금액+1) (분류등급별, 2022~2025 통합)")
ax.set_xlabel("월")
ax.set_ylabel("평균 log(매출금액 + 1)")
ax.set_xticks(range(1, 13))
ax.legend()
plt.tight_layout()
plt.show()

## 2-1-5. 데이터 누락 구간 확인

도시별로 2022-01-01 ~ 2025-12-31 기간 중 실제 데이터가 있는 날짜가 몇 %인지,
연속으로 비는 구간(7일 이상)이 있는지 확인한다.

In [ ]:

# 기준이 되는 전체 날짜 범위 생성

# 데이터프레임의 최소 날짜부터 최대 날짜까지 하루 단위(freq="D")로 빠짐없는 전체 날짜 배열을 생성
full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
print(f"전체 기간: {len(full_range)}일\n")


# 도시별 데이터 커버리지 및 누락 일자 계산
for city in sorted(df["도시"].unique()):
    # 현재 도시의 데이터에 존재하는 실제 날짜들을 중복 없이 추출하여 집합(set) 자료형으로 만듬
    city_dates = set(df.loc[df["도시"] == city, "date"].unique())
    # [차집합 연산]
    missing_dates = sorted(set(full_range) - city_dates)
    # [커버리지 계산]
    coverage = (1 - len(missing_dates) / len(full_range)) * 100
    # 도시별 최종 커버리지 비율과 총 누락 일수를 출력
    print(f"[{city}] 커버리지 {coverage:.1f}% (누락 {len(missing_dates)}일)")

    
    # 연속 누락 구간(Gap) 탐지 및 장기 누락 출력
    if missing_dates:
        # 연속 누락 구간(7일 이상)만 추려서 출력
        gaps = [] # 탐지된 (시작일, 종료일) 묶음들을 담을 리스트
        start = missing_dates[0]   # 연속 누락 구간의 시작점 초기화
        prev = missing_dates[0]    # 직전 날짜 기록용 변수 초기화
        
        # 두 번째 누락 일자부터 순회하면서 연속성이 끊어지는 지점 찾기
        for d in missing_dates[1:]:
            # 현재 누락일(d)과 직전 누락일(prev)의 차이가 1일보다 크다면 연속성이 끊긴 것
            if (d - prev).days > 1:
                gaps.append((start, prev))
                start = d
            prev = d
        gaps.append((start, prev))

        # 탐지된 구간 중 누락 기간이 '7일 이상'인 장기 연속 누락 구간만 따로 추리기
        long_gaps = [(s, e) for s, e in gaps if (e - s).days + 1 >= 7]
        # 7일 이상 연속 누락된 구간의 시작일, 종료일, 그리고 누락 일수를 포맷팅하여 출력
        for s, e in long_gaps:
            print(f"    - {s.date()} ~ {e.date()} ({(e - s).days + 1}일 연속 누락)")